In [ ]:
# !!! ATTENZIONE !!!
# modificare i path
annotations_path = "/inserisci/il/path/per/pp4av/annotations"
images_path = "/inserisci/il/path/per/pp4av/images"
output_dir = "/inserisci/il/path/per/output"

## Split 1: Primo ciclo sperimentale

In [ ]:
import os
import shutil
import random

# --- 1. PULIZIA AMBIENTE ---
if os.path.exists(output_dir):
    print("Pulizia cartella di lavoro precedente in corso...")
    shutil.rmtree(output_dir)

# --- 2. CREAZIONE CARTELLE ---
splits = ["train", "val", "test_normal", "test_fisheye"]
for split in splits:
    os.makedirs(os.path.join(output_dir, "images", split), exist_ok = True)
    os.makedirs(os.path.join(output_dir, "labels", split), exist_ok = True)

# --- 3. FUNZIONE DI COPIA ---
def copy_files(file_list, split_name, city_img_dir, city_lbl_dir):
    copied = 0
    for txt in file_list:
        base_name = os.path.splitext(txt)[0]
        # Copia label
        shutil.copy(os.path.join(city_lbl_dir, txt), os.path.join(output_dir, "labels", split_name, txt))
        # Trova e copia immagine
        img_path = os.path.join(city_img_dir, base_name + ".png")
        if not os.path.exists(img_path):
            img_path = os.path.join(city_img_dir, base_name + ".png")
        if os.path.exists(img_path):
            shutil.copy(img_path, os.path.join(output_dir, "images", split_name, os.path.basename(img_path)))
            copied += 1
    return copied

# --- 4. LOGICA DI SPLIT ---
print("Inizio suddivisione dei dati...")
cities = [d for d in os.listdir(annotations_path) if os.path.isdir(os.path.join(annotations_path, d))]
counts = {"train": 0, "val": 0, "test_normal": 0, "test_fisheye": 0}

# Aggiunta del seed per la riproducibilità scientifica dell'esperimento
random.seed(42)

for city in cities:
    city_img_dir = os.path.join(images_path, city)
    city_lbl_dir = os.path.join(annotations_path, city)
    txt_files = [f for f in os.listdir(city_lbl_dir) if f.endswith(".txt")]
    
    # Se è Fisheye, va TUTTO nel test_fisheye
    if city.lower() == "fisheye":
        counts["test_fisheye"] += copy_files(txt_files, "test_fisheye", city_img_dir, city_lbl_dir)
        print(f"[{city.upper()}] -> 100% in Test Fisheye: {len(txt_files)} file")
        continue

    # Se sono immagini normali, facciamo lo split 80-10-10
    random.shuffle(txt_files)
    total_files = len(txt_files)
    train_end = int(total_files * 0.8)
    val_end = train_end + int(total_files * 0.1)
    
    train_files = txt_files[:train_end]
    val_files = txt_files[train_end:val_end]
    test_files = txt_files[val_end:]
    
    counts["train"] += copy_files(train_files, "train", city_img_dir, city_lbl_dir)
    counts["val"] += copy_files(val_files, "val", city_img_dir, city_lbl_dir)
    counts["test_normal"] += copy_files(test_files, "test_normal", city_img_dir, city_lbl_dir)
    
    print(f"[{city.upper()}] -> Train: {len(train_files)} | Val: {len(val_files)} | Test Normal: {len(test_files)}")

print("\n=== SPLIT COMPLETATO CON SUCCESSO ===")
print(f"Totale Immagini in TRAIN: {counts['train']}")
print(f"Totale Immagini in VAL (per addestramento): {counts['val']}")
print(f"Totale Immagini in TEST NORMAL (Colonna Sinistra): {counts['test_normal']}")
print(f"Totale Immagini in TEST FISHEYE (Colonna Destra): {counts['test_fisheye']}")

In [ ]:
import yaml

# --- 5. CREAZIONE FILE YAML ---
# Yaml per il Training
yaml_train = {
    "path": output_dir, "train": "images/train", "val": "images/val",
    "nc": 2, "names": ["face", "license_plate"]
}
# Yaml per Test Immagini Normali
yaml_test_normal = {
    "path": output_dir, "train": "images/train", "val": "images/test_normal",
    "nc": 2, "names": ["face", "license_plate"]
}
# Yaml per Test Immagini Fisheye
yaml_test_fisheye = {
    "path": output_dir, "train": "images/train", "val": "images/test_fisheye",
    "nc": 2, "names": ["face", "license_plate"]
}

# !!! ATTENZIONE !!!
# modificare i path
with open("/inserisci/il/path/di/output/per/dataset_train.yaml", 'w') as f: yaml.dump(yaml_train, f, sort_keys = False)
with open("/inserisci/il/path/di/output/per/dataset_test_normal.yaml", 'w') as f: yaml.dump(yaml_test_normal, f, sort_keys = False)
with open("/inserisci/il/path/di/output/per/dataset_test_fisheye.yaml", 'w') as f: yaml.dump(yaml_test_fisheye, f, sort_keys = False)

print("\nFile YAML generati: 'dataset_train.yaml', 'dataset_test_normal.yaml', 'dataset_test_fisheye.yaml'")